# MVP04 — Camada Silver

Lê a camada `bronze`, diagnostica a qualidade dos dados, aplica os tratamentos
necessários e persiste o resultado na camada `silver`.

**Posição no pipeline:** quarto notebook. Requer `MVP03-Bronze` executado.

**Princípio da camada:** a Silver mantém a granularidade da origem, mas entrega o dado
limpo, tipado e confiável. É aqui que os problemas de qualidade são identificados e
resolvidos — nenhuma camada posterior precisa tratá-los de novo.

## Estrutura do notebook

| Etapa | Conteúdo |
|---|---|
| 1 | Parâmetros e funções auxiliares |
| 2 | **Diagnóstico de qualidade sobre a Bronze** — antes de qualquer tratamento |
| 3 | Transformações e correções, tabela por tabela |
| 4 | **Validação pós-tratamento** — os mesmos testes, agora sobre a Silver |
| 5 | Documentação no Unity Catalog |
| 6 | Resumo do tratamento aplicado |

Rodar os mesmos testes nas etapas 2 e 4 evidencia tanto o problema encontrado quanto a sua resolução.

## 📚 Sobre este Notebook

Este notebook implementa a ingestão da **camada Silver** a partir dos dados já carregados na **camada Bronze**.

### Arquitetura Medalhão

* **📂 Staging**: Dados brutos copiados do Google Drive (notebook MVP02-Download)
* **🥉 Bronze**: Dados brutos com metadados de rastreabilidade (notebook MVP03-Bronze)
* **🥈 Silver**: Dados limpos, padronizados e com tipos corretos (este notebook)
* **🥇 Gold**: Dados agregados e prontos para consumo  (notebook MVP05-Gold)

### 🎯 Objetivo
Ler as tabelas da camada Bronze e aplicnar as seguintes transformações:

* **Normalização de nomes**: colunas em lowercase com separador `_`
* **Conversão de tipos**: strings de data → `DATE`/`TIMESTAMP`, strings numéricas → `DOUBLE`/`INT`
* **Tratamento de valores especiais**: `-` → `NULL`
* **Decimais**: trocar vírgula por ponto (padrão brasileiro → padrão internacional)
* **Metadados de rastreabilidade**: `data_carga_silver` e `origem_tabela_bronze`

In [0]:
%sql
-- Define o catálogo ativo como mvpDataEngineer no Unity Catalog.
USE CATALOG mvpDataEngineer;

-- Define o schema ativo como silver dentro do catálogo atual.
USE SCHEMA silver;

## 🔧 Etapa 1 — Parâmetros e funções auxiliares

### Colunas descartadas nesta camada

Cinco colunas presentes nas três tabelas de origem **não seguem para a Silver**, por não
terem valor analítico:

| Coluna | Motivo do descarte |
|---|---|
| `PROTOCOLO` | Identificador do formulário de origem. É metadado do sistema de lançamento, não do fenômeno medido. Nenhuma pergunta de negócio depende dele. |
| `SOLICDATA` | Data do lançamento no sistema, não da medição. Irrelevante para a análise. |
| `STATUS` | Situação no fluxo de aprovação. **Usado como filtro antes de ser descartado** — ver abaixo. |
| `UNIDADE` | Constante em todas as linhas das três tabelas. Coluna sem variação não carrega informação. |
| `OBSERVACOES` | Texto livre, vazio em duas das três fontes e preenchido em 4 de 226 linhas na terceira. Volume insuficiente para tratamento estruturado. |

**`PROTOCOLO` e `SOLICDATA` são usados durante o processamento e descartados ao final.**
Eles são necessários para dois tratamentos: `SOLICDATA` serve de critério de desempate na
deduplicação (o lançamento mais recente corrige o anterior), e `PROTOCOLO` permite
identificar registros atribuídos à entidade ou à data errada. Cumprida essa função, saem.

**`STATUS` é aplicado como filtro antes do descarte.** Apenas registros com
`Solicitação Finalizada` — homologados pela operação — seguem adiante. Depois do filtro,
a coluna passa a ter valor único e é removida pelo mesmo motivo de `UNIDADE`.

In [0]:
CATALOGO = "mvpdataengineer"

STATUS_HOMOLOGADO = "Solicitação Finalizada"

print(f"Catálogo: {CATALOGO}")
print(f"Status homologado: {STATUS_HOMOLOGADO}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def texto_para_numero(coluna, tipo="double"):
    """Converte texto da origem em número.

    Trata as duas formas de ausência presentes nos arquivos (hífen e célula vazia)
    e a vírgula como separador decimal.
    """
    limpo = F.trim(F.col(coluna))
    return (F.when(limpo.isin("-", ""), None)
             .otherwise(F.regexp_replace(limpo, ",", ".").cast(tipo)))


def chave_duplicada(df, chaves, medidas, col_protocolo="PROTOCOLO", col_solicdata="SOLICDATA"):
    """Retorna as chaves de negócio com mais de um registro.

    A CHAVE é a entidade medida mais a DATA DO REGISTRO — a data do fato ocorrido em
    campo. `PROTOCOLO` e `SOLICDATA` NÃO fazem parte da chave: eles identificam o
    lançamento no sistema, não a medição. Dois protocolos lançados em dias diferentes
    que descrevem a mesma leitura da mesma data são exatamente a duplicidade a
    encontrar, e incluí-los na chave a esconderia.

    Também não se agrupa pelas medidas. Agrupar por elas acharia apenas linhas
    idênticas e deixaria passar o caso pior: mesma chave com valores divergentes.

    A classificação resultante:
      - REENVIO           : mesma leitura lançada duas vezes. Deduplicação segura.
      - CONFLITO DE VALOR : mesma chave com valores diferentes. Exige decisão explícita.
    """
    agg = [
        F.count("*").alias("lancamentos"),
        F.collect_set(F.col(col_protocolo).cast("string")).alias("protocolos"),
        F.sort_array(F.collect_set(F.col(col_solicdata).cast("string"))).alias("datas_lancamento"),
    ]
    for m in medidas:
        agg.append(F.countDistinct(m).alias(f"distintos_{m}"))
        agg.append(F.collect_set(F.col(m).cast("string")).alias(f"valores_{m}"))

    # greatest() exige ao menos duas colunas; com uma única medida, usa-se a própria.
    cols_distintos = [F.col(f"distintos_{m}") for m in medidas]
    max_distintos = cols_distintos[0] if len(cols_distintos) == 1 else F.greatest(*cols_distintos)

    return (df.groupBy(*chaves).agg(*agg)
              .filter(F.col("lancamentos") > 1)
              .withColumn("classificacao",
                          F.when(max_distintos > 1, F.lit("CONFLITO DE VALOR"))
                           .otherwise(F.lit("REENVIO")))
              .orderBy(F.desc("classificacao"), *chaves))


def resumo_duplicidade(df, chaves, medidas, rotulo):
    """
    Imprime o placar de duplicidade de uma tabela.
    """
    d = chave_duplicada(df, chaves, medidas)
    total_chaves = df.select(*chaves).distinct().count()
    contagem = {r["classificacao"]: r["qtd"] for r in
                d.groupBy("classificacao").agg(F.count("*").alias("qtd")).collect()}
    reenvio = contagem.get("REENVIO", 0)
    conflito = contagem.get("CONFLITO DE VALOR", 0)
    excedentes = df.count() - total_chaves

    print(f"{rotulo}")
    print(f"  Chave de negócio .................: {chaves}")
    print(f"  Registros na tabela ..............: {df.count()}")
    print(f"  Chaves distintas .................: {total_chaves}")
    print(f"  Lançamentos excedentes ...........: {excedentes}")
    print(f"  Chaves com reenvio ...............: {reenvio}")
    print(f"  Chaves com conflito de valor .....: {conflito}")
    return d


def perfil_completude(df, colunas):
    """Percentual de nulos por coluna."""
    total = df.count()
    linhas = []
    for c in colunas:
        nulos = df.filter(F.col(c).isNull()).count()
        linhas.append({
            "coluna": c,
            "total": total,
            "nulos": nulos,
            "pct_nulos": round(100.0 * nulos / total, 2) if total else None,
        })
    return spark.createDataFrame(linhas).orderBy(F.desc("pct_nulos"))


print("Funções auxiliares definidas.")

# 🔍 Etapa 2 — Diagnóstico de qualidade

Verificação executada **sobre a camada Bronze**, antes de qualquer tratamento. Cada
tabela é avaliada nas cinco dimensões de qualidade: completude, consistência, unicidade,
acurácia e presença de outliers.

As células desta etapa são diagnósticas: não alteram nenhum dado.

In [0]:
bronze_pluv  = spark.table(f"{CATALOGO}.bronze.leitura_pluviometros")
bronze_bacia = spark.table(f"{CATALOGO}.bronze.medicao_bacias")
bronze_laje  = spark.table(f"{CATALOGO}.bronze.medicao_laje")

print("Volume na Bronze")
print(f"  leitura_pluviometros : {bronze_pluv.count():>6} registros")
print(f"  medicao_bacias       : {bronze_bacia.count():>6} registros")
print(f"  medicao_laje         : {bronze_laje.count():>6} registros")

In [0]:
# Consistência — domínio dos campos categóricos
# Verifica se os campos de domínio fechado contêm apenas os valores esperados. Variação de grafia em um campo categórico quebra junções e duplica membros de dimensão.

print("STATUS — medicao_bacias")
bronze_bacia.groupBy("STATUS").count().show(truncate=False)

print("STATUS — leitura_pluviometros")
bronze_pluv.groupBy("STATUS").count().show(truncate=False)

print("STATUS — medicao_laje")
bronze_laje.groupBy("STATUS").count().show(truncate=False)

print("UNIDADE — valores distintos por tabela")
for nome, df in [("pluviometros", bronze_pluv), ("bacias", bronze_bacia), ("laje", bronze_laje)]:
    print(f"  {nome}: {[r[0] for r in df.select('UNIDADE').distinct().collect()]}")

In [0]:
# Rótulos das estações pluviométricas
# A salina tem cinco estações. Se aparecerem mais de cinco rótulos, há variação de grafia.
display(
    bronze_pluv.groupBy("ESTACAO")
        .agg(F.count("*").alias("registros"),
             F.min("DATAREGISTRO").alias("primeiro_registro"),
             F.max("DATAREGISTRO").alias("ultimo_registro"))
        .orderBy("ESTACAO")
)

In [0]:
# Identificadores das bacias

display(bronze_bacia.groupBy("BACIA").agg(F.count("*").alias("registros")).orderBy("BACIA"))

In [0]:
# Consistência — representação de valores ausentes
# Nos arquivos de origem a ausência de valor aparece de duas formas: o caractere `-` e a célula vazia. Ambas precisam ser convertidas para `NULL`, e a contagem abaixo dimensiona o problema.

colunas_numericas_bacia = ["AGUADOCE_COLOCADO", "AGUADOCE_RETIRADO",
                           "SALMOURA_COLOCADO", "SALMOURA_RETIRADO", "SALMOURA_DENSIDADE"]

linhas = []
for c in colunas_numericas_bacia:
    linhas.append({
        "tabela": "medicao_bacias",
        "coluna": c,
        "hifen": bronze_bacia.filter(F.trim(F.col(c)) == "-").count(),
        "vazio_ou_nulo": bronze_bacia.filter(F.col(c).isNull() | (F.trim(F.col(c)) == "")).count(),
        "preenchido": bronze_bacia.filter(
            F.col(c).isNotNull() & (F.trim(F.col(c)) != "-") & (F.trim(F.col(c)) != "")).count(),
    })

linhas.append({
    "tabela": "leitura_pluviometros",
    "coluna": "MILIMETROS",
    "hifen": bronze_pluv.filter(F.trim(F.col("MILIMETROS")) == "-").count(),
    "vazio_ou_nulo": bronze_pluv.filter(
        F.col("MILIMETROS").isNull() | (F.trim(F.col("MILIMETROS")) == "")).count(),
    "preenchido": bronze_pluv.filter(
        F.col("MILIMETROS").isNotNull() & (F.trim(F.col("MILIMETROS")) != "-")
        & (F.trim(F.col("MILIMETROS")) != "")).count(),
})

display(spark.createDataFrame(linhas))

## 2.1 Unicidade — mesma medição lançada mais de uma vez

### Definição da chave de negócio

A chave é a **entidade medida mais a data do registro** — a data em que a medição
ocorreu em campo:

| Tabela | Chave de negócio |
|---|---|
| `leitura_pluviometros` | (`ESTACAO`, `DATAREGISTRO`) |
| `medicao_bacias` | (`BACIA`, `DATAREGISTRO`) |
| `medicao_laje` | (`CRISTALIZADOR`, `DATAMEDICAO`) |

**`PROTOCOLO` e `SOLICDATA` ficam fora da chave**, e é isso que faz o teste funcionar.
Eles identificam o *lançamento no sistema*, não a *medição*. O caso típico:

| Protocolo | SOLICDATA | DATAREGISTRO | Milímetros |
|---|---|---|---|
| 215487 | 01/07/2026 | 01/07/2026 | 12,5 |
| 215479 | 02/07/2026 | 01/07/2026 | 12,5 |

São duas inclusões no sistema, em dias diferentes, descrevendo **a mesma leitura de
12,5 mm do dia 01/07**. Existe uma única medição e duas linhas. Se `PROTOCOLO` ou
`SOLICDATA` entrassem na chave, as duas linhas pareceriam registros distintos e a
duplicidade passaria despercebida — que é exatamente o erro a evitar.

### Classificação

| Classificação | Significado | Ação |
|---|---|---|
| `REENVIO` | Mesma leitura lançada mais de uma vez, com valores iguais | Deduplicação segura: manter uma linha |
| `CONFLITO DE VALOR` | Mesma chave com valores diferentes | Exige critério explícito de qual prevalece |

As colunas `protocolos` e `datas_lancamento` no resultado mostram quais lançamentos
originaram cada duplicidade.

In [0]:
# Pluviômetros — chave (estação, data de registro)

dup_pluv = resumo_duplicidade(bronze_pluv, ["ESTACAO", "DATAREGISTRO"], ["MILIMETROS"],"leitura_pluviometros")
display(dup_pluv)

In [0]:
# Bacias — chave (bacia, data de registro)

dup_bacia = resumo_duplicidade(bronze_bacia, ["BACIA", "DATAREGISTRO"],["AGUADOCE_COLOCADO", "SALMOURA_COLOCADO", "SALMOURA_DENSIDADE"],"medicao_bacias")
display(dup_bacia)

In [0]:
# Laje — chave (cristalizador, data de medição)

dup_laje = resumo_duplicidade(bronze_laje, ["CRISTALIZADOR", "DATAMEDICAO"],
                              ["REGUA_1", "REGUA_2", "REGUA_3", "REGUA_4","REGUA_5", "REGUA_6", "REGUA_7", "REGUA_8"],
                              "medicao_laje")
display(dup_laje)

### Detalhamento das linhas duplicadas

Exibe as linhas completas por trás de cada chave duplicada, lado a lado, para que a
decisão de qual registro manter seja tomada com o dado à vista.

In [0]:
chaves_dup_laje = dup_laje.select("CRISTALIZADOR", "DATAMEDICAO")

display(bronze_laje.join(chaves_dup_laje, ["CRISTALIZADOR", "DATAMEDICAO"])
        .select("CRISTALIZADOR", "DATAMEDICAO", "PROTOCOLO", "SOLICDATA",
                *[f"REGUA_{i}" for i in range(1, 9)])
        .orderBy("CRISTALIZADOR", "DATAMEDICAO", "SOLICDATA"))

In [0]:
chaves_dup_pluv = dup_pluv.select("ESTACAO", "DATAREGISTRO")

display(bronze_pluv.join(chaves_dup_pluv, ["ESTACAO", "DATAREGISTRO"])
        .select("ESTACAO", "DATAREGISTRO", "PROTOCOLO", "SOLICDATA", "MILIMETROS")
        .orderBy("DATAREGISTRO", "ESTACAO", "SOLICDATA"))

In [0]:
chaves_dup_bacia = dup_bacia.select("BACIA", "DATAREGISTRO")

display(bronze_bacia.join(chaves_dup_bacia, ["BACIA", "DATAREGISTRO"])
        .select("BACIA", "DATAREGISTRO", "PROTOCOLO", "SOLICDATA",
                "AGUADOCE_COLOCADO", "SALMOURA_COLOCADO", "SALMOURA_DENSIDADE")
        .orderBy("DATAREGISTRO", "BACIA", "SOLICDATA"))

In [0]:
# A medição é diária. Dias faltantes em uma bacia e presentes na outra reforçam o diagnóstico de registro faltante.

dias_por_bacia = (bronze_bacia
    .groupBy("BACIA")
    .agg(F.countDistinct("DATAREGISTRO").alias("dias_com_registro"),
         F.min("DATAREGISTRO").alias("primeira_data"),
         F.max("DATAREGISTRO").alias("ultima_data")))

display(dias_por_bacia)

datas_b1 = {r[0] for r in bronze_bacia.filter(F.col("BACIA") == "BACIA1").select("DATAREGISTRO").distinct().collect()}
datas_b2 = {r[0] for r in bronze_bacia.filter(F.col("BACIA") == "BACIA2").select("DATAREGISTRO").distinct().collect()}

print(f"Datas presentes na BACIA1 e ausentes na BACIA2: {sorted(datas_b1 - datas_b2)}")
print(f"Datas presentes na BACIA2 e ausentes na BACIA1: {sorted(datas_b2 - datas_b1)}")

## 2.2 Acurácia — faixa de valores

Verifica se os valores numéricos fazem sentido no contexto físico do processo.

In [0]:
# Verificação de reguas com leitura máxima

laje_num = bronze_laje.select(
    texto_para_numero("CRISTALIZADOR", "int").alias("cristalizador"),
    F.col("DATAMEDICAO").alias("data_medicao"),
    *[texto_para_numero(c, "int").alias(c.lower()) for c in reguas])

cols_r = [F.col(c.lower()) for c in reguas]

display(laje_num
    .withColumn("reguas_em_100", sum([F.when(c == 100, 1).otherwise(0) for c in cols_r]))
    .filter(F.col("reguas_em_100") > 0)
    .orderBy(F.desc("reguas_em_100")))

------------------------------------
## SEPARAÇÃO DE TRABALHO
------------------------------------

## 🥈 Etapa 1: Transformação Bronze → Silver

Lê cada tabela da camada **Bronze**, aplica limpeza e padronização, e grava na camada **Silver**.

### Transformações aplicadas:
* **Normalização de nomes**: colunas em lowercase com `_` (ex: `DATAREGISTRO` → `data_registro`)
* **Conversão de datas**: strings `dd/MM/yyyy` → `DATE` e `dd/MM/yyyy HH:mm:ss` → `TIMESTAMP`
* **Conversão de decimais**: vírgula → ponto (ex: `14,0` → `14.0` → `DOUBLE`)
* **Tratamento de valores ausentes**: `-` → `NULL`
* **Metadados de rastreabilidade**: `data_carga_silver` e `origem_tabela_bronze`
* **Aplicado filtro**: `STATUS = Solicitação Finalizada`
* **Campos removidos**: `OBSERVACAO` por não trazer informações significativas para analise e `SOLICDATA` pois é o momento que a leitura foi inserida no sistema. O campo `DATAREGISTRO` é o momento que a leitura foi realizada em campo e este é importante para as análises.

### Princípios da camada Silver:
* ✅ **Dados limpos** - tipos corretos, sem valores inválidos
* ✅ **Nomes padronizados** - lowercase com snake_case
* ✅ **Preserva rastreabilidade** - mantém metadados de staging e bronze
* ✅ **Formato Delta Lake** - formato otimizado do Databricks

In [0]:
# importando bibliotecas 
from pyspark.sql.functions import col, to_timestamp, to_date, regexp_replace, current_timestamp, lit, when

In [0]:
# =========================================
# TRANSFORMAÇÃO: leitura_pluviometros
# =========================================
# Transformações aplicadas:
#   - DATAREGISTRO: STRING "dd/MM/yyyy" para DATE
#   - MILIMETROS: vírgula decimal para ponto e STRING para DOUBLE
#   - Nomes de colunas: uppercase para lowercase com _
#   - Removido campos: OBSERVACAO e SOLICDATA. 
#   - Aplicado filtro para STATUS = Solicitação Finalizada

# Ler da camada Bronze
df_bronze = spark.table("mvpDataEngineer.bronze.leitura_pluviometros")
print(f" Registros lidos da bronze: {df_bronze.count()}")

# Filta apenas os registros cujo status for "Solicitação Finalizada"
df_bronze = df_bronze.filter(col("STATUS") == "Solicitação Finalizada")

# Aplicar transformações
df_silver = df_bronze.select(
    col("UNIDADE").alias("unidade"),
    col("ESTACAO").alias("estacao"),
    to_date(col("DATAREGISTRO"), "dd/MM/yyyy").alias("data_registro"),
    when(col("MILIMETROS") == "-", None).otherwise(regexp_replace(col("MILIMETROS"), ",", ".").cast("double")).alias("milimetros"),
    # Metadados de rastreabilidade (Staging + Bronze)
    col("data_ingestao"),
    col("origem"),
    col("arquivo_origem"),
    col("data_carga_bronze"),
    col("origem_tabela"),
    # Metadados de rastreabilidade (Silver)
    current_timestamp().alias("data_carga_silver"),
    lit("bronze.leitura_pluviometros").alias("origem_tabela_bronze")
)

# Salvar na camada Silver
df_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("mvpDataEngineer.silver.leitura_pluviometros")

row_count = df_silver.count()
print(f"  -Tabela criada: mvpDataEngineer.silver.leitura_pluviometros")
print(f"  -Registros salvos: {row_count}")
print(f"  -Colunas: {len(df_silver.columns)}")

In [0]:
# =========================================
# TRANSFORMAÇÃO: medicao_bacias
# =========================================
# Transformações aplicadas:
#   - SOLICDATA de STRING "dd/MM/yyyy HH:mm:ss" para TIMESTAMP
#   - DATAREGISTRO de STRING "dd/MM/yyyy" para DATE
#   - AGUADOCE_COLOCADO, AGUADOCE_RETIRADO, SALMOURA_COLOCADO e SALMOURA_RETIRADO transformado "-" em NULL, depois STRING  para INT
#   - SALMOURA_DENSIDADE: vírgula decimal para ponto e STRING para DOUBLE
#   - Nomes de colunas: uppercase para lowercase com _
#   - Removido campo de observeção
#   - Aplicado filtro para STATUS = Solicitação Finalizada

# Ler da camada Bronze
df_bronze = spark.table("mvpDataEngineer.bronze.medicao_bacias")
print(f" Registros lidos da bronze: {df_bronze.count()}")

# Filta apenas os registros cujo status for "Solicitação Finalizada"
df_bronze = df_bronze.filter(col("STATUS") == "Solicitação Finalizada")

df_silver = df_bronze.select(
    col("UNIDADE").alias("unidade"),
    col("BACIA").alias("bacia"),
    to_date(col("DATAREGISTRO"), "dd/MM/yyyy").alias("data_registro"),
    # Tratar "-" como NULL e converter para INT
    when(col("AGUADOCE_COLOCADO") == "-", None).otherwise(col("AGUADOCE_COLOCADO").cast("int")).alias("agua_doce_colocado"),
    when(col("AGUADOCE_RETIRADO") == "-", None).otherwise(col("AGUADOCE_RETIRADO").cast("int")).alias("agua_doce_retirado"),
    when(col("SALMOURA_COLOCADO") == "-", None).otherwise(col("SALMOURA_COLOCADO").cast("int")).alias("salmoura_colocado"),
    when(col("SALMOURA_RETIRADO") == "-", None).otherwise(col("SALMOURA_RETIRADO").cast("int")).alias("salmoura_retirado"),
    # Densidade: vírgula decimal para ponto e transforma em DOUBLE
    when(col("SALMOURA_DENSIDADE") == "-", None).otherwise(regexp_replace(col("SALMOURA_DENSIDADE"), ",", ".").cast("double")).alias("salmoura_densidade"),
    # Metadados de rastreabilidade (Staging + Bronze)
    col("data_ingestao"),
    col("origem"),
    col("arquivo_origem"),
    col("data_carga_bronze"),
    col("origem_tabela"),
    # Metadados de rastreabilidade (Silver)
    current_timestamp().alias("data_carga_silver"),
    lit("bronze.medicao_bacias").alias("origem_tabela_bronze")
)

# Salvar na camada Silver
df_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("mvpDataEngineer.silver.medicao_bacias")

row_count = df_silver.count()
print(f"  -Tabela criada: mvpDataEngineer.silver.leitura_pluviometros")
print(f"  -Registros salvos: {row_count}")
print(f"  -Colunas: {len(df_silver.columns)}")

In [0]:
# =========================================
# TRANSFORMAÇÃO: medicao_laje
# =========================================
# Transformações aplicadas:
#   - SOLICDATA de STRING "dd/MM/yyyy HH:mm:ss" para TIMESTAMP
#   - DATAMEDICAO: string "dd/MM/yyyy" para DATE
#   - REGUA_1 a REGUA_8: já estão como INT, apenas renomeados
#   - Nomes de colunas: uppercase para lowercase com _
#   - Removido campo de observeção
#   - Aplicado filtro para STATUS = Solicitação Finalizada

# Ler da camada Bronze
df_bronze = spark.table("mvpDataEngineer.bronze.medicao_laje")
print(f" Registros lidos da bronze: {df_bronze.count()}")

# Filta apenas os registros cujo status for "Solicitação Finalizada"
df_bronze = df_bronze.filter(col("STATUS") == "Solicitação Finalizada")

# Aplicar transformações
df_silver = df_bronze.select(
    col("UNIDADE").alias("unidade"),
    to_date(col("DATAMEDICAO"), "dd/MM/yyyy").alias("data_medicao"),
    col("CRISTALIZADOR").alias("cristalizador"),
    col("AREAHECTARES").alias("area_hectares"),
    col("REGUA_1").alias("regua_1"),
    col("REGUA_2").alias("regua_2"),
    col("REGUA_3").alias("regua_3"),
    col("REGUA_4").alias("regua_4"),
    col("REGUA_5").alias("regua_5"),
    col("REGUA_6").alias("regua_6"),
    col("REGUA_7").alias("regua_7"),
    col("REGUA_8").alias("regua_8"),
    # Metadados de rastreabilidade (Staging + Bronze)
    col("data_ingestao"),
    col("origem"),
    col("arquivo_origem"),
    col("data_carga_bronze"),
    col("origem_tabela"),
    # Metadados de rastreabilidade (Silver)
    current_timestamp().alias("data_carga_silver"),
    lit("bronze.medicao_laje").alias("origem_tabela_bronze")
)

# Salvar na camada Silver
df_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("mvpDataEngineer.silver.medicao_laje")

row_count = df_silver.count()
print(f"  -Tabela criada: mvpDataEngineer.silver.leitura_pluviometros")
print(f"  -Registros salvos: {row_count}")
print(f"  -Colunas: {len(df_silver.columns)}")

## 🔍 Etapa 2: Validação dos Dados Silver

### O que verificar?
* **Total de registros** por tabela (deve ser igual à Bronze)
* **Tipos de dados** convertidos corretamente
* **Valores NULL** onde antes havia `-`
* **Amostras** dos dados transformados

In [0]:
%sql
-- Resumo das tabelas Silver: total de registros e última carga
SELECT 
  'LEITURA_PLUVIOMETROS' as tabela,
  COUNT(*) as total_registros,
  MAX(data_carga_silver) as ultima_carga_silver
FROM silver.leitura_pluviometros

UNION ALL

SELECT 
  'MEDICAO_BACIAS' as tabela,
  COUNT(*) as total_registros,
  MAX(data_carga_silver) as ultima_carga_silver
FROM silver.medicao_bacias

UNION ALL

SELECT 
  'MEDICAO_LAJE' as tabela,
  COUNT(*) as total_registros,
  MAX(data_carga_silver) as ultima_carga_silver
FROM silver.medicao_laje

ORDER BY tabela;

In [0]:
%sql
-- Amostra de leitura_pluviometros (primeiras 10 linhas)
-- Verificar: 
--      data_registro como DATE, 
--      milimetros como DOUBLE, 
--      nomes em lowercase

SELECT unidade, estacao, data_registro, milimetros
FROM mvpDataEngineer.silver.leitura_pluviometros LIMIT 10;

In [0]:
%sql
-- Amostra de medicao_bacias (primeiras 10 linhas)
-- Verificar: 
--      valores "-" agora são NULL, 
--      salmoura_densidade como DOUBLE

SELECT bacia, data_registro, agua_doce_colocado, agua_doce_retirado, 
       salmoura_colocado, salmoura_retirado, salmoura_densidade
FROM mvpDataEngineer.silver.medicao_bacias LIMIT 10;

In [0]:
%sql
-- Amostra de medicao_laje (primeiras 10 linhas)
-- Verificar: data_medicao como DATE, nomes das réguas em lowercase
SELECT data_medicao, cristalizador, area_hectares, 
       regua_1, regua_2, regua_3, regua_4
FROM mvpDataEngineer.silver.medicao_laje LIMIT 10;

In [0]:
%sql
-- Identificar registros duplicados na tabela silver.leitura_pluviometros
-- Agrupa por todas as colunas de negócio e conta quantas vezes cada combinação aparece
SELECT 
  unidade,
  estacao,
  data_registro,
  milimetros,
  COUNT(*) AS qtd_ocorrencias
FROM mvpDataEngineer.silver.leitura_pluviometros
GROUP BY 
  unidade, estacao, data_registro, milimetros
HAVING COUNT(*) > 1
ORDER BY qtd_ocorrencias DESC;

In [0]:
%sql
-- Identificar registros duplicados na tabela silver.medicao_bacias
-- Agrupa por todas as colunas de negócio e conta quantas vezes cada combinação aparece
SELECT 
  unidade,
  bacia,
  data_registro,
  agua_doce_colocado,
  agua_doce_retirado,
  salmoura_colocado,
  salmoura_retirado,
  salmoura_densidade,
  COUNT(*) AS qtd_ocorrencias
FROM mvpDataEngineer.silver.medicao_bacias
GROUP BY 
  unidade, bacia, data_registro, 
  agua_doce_colocado, agua_doce_retirado, salmoura_colocado, salmoura_retirado, salmoura_densidade
HAVING COUNT(*) > 1
ORDER BY qtd_ocorrencias DESC;

In [0]:
%sql
-- Identificar registros duplicados na tabela silver.medicao_laje
-- Agrupa por todas as colunas de negócio e conta quantas vezes cada combinação aparece
SELECT 
  unidade,
  data_medicao,
  cristalizador,
  area_hectares,
  regua_1,  regua_2,  regua_3,  regua_4,  regua_5,  regua_6,  regua_7,  regua_8,
  COUNT(*) AS qtd_ocorrencias
FROM mvpDataEngineer.silver.medicao_laje
GROUP BY 
  unidade, data_medicao, cristalizador, area_hectares,
  regua_1, regua_2, regua_3, regua_4, regua_5, regua_6, regua_7, regua_8
HAVING COUNT(*) > 1
ORDER BY qtd_ocorrencias DESC;

In [0]:
# =========================================
# DEDUPLICAÇÃO: medicao_laje
# =========================================
# Regra de negócio: só pode existir uma leitura por dia (data_medicao) por cristalizador.
# Foram identificados registros duplicados (mais de um protocolo com a mesma leitura).
# Solução: manter apenas um registro por combinação (data_medicao, cristalizador),
# preferindo o protocolo de menor número (mais antigo).

from pyspark.sql.functions import row_number, count as count_fn
from pyspark.sql.window import Window

# Ler a tabela Silver já gravada
df_silver = spark.table("mvpDataEngineer.silver.medicao_laje")

total_antes = df_silver.count()
print(f"Registros antes da deduplicação: {total_antes}")

# Janela particionada por data_medicao + cristalizador, ordenada por protocolo
# para manter sempre o mesmo registro (protocolo de menor número)
w = Window.partitionBy("data_medicao", "cristalizador").orderBy("protocolo")

df_dedup = (df_silver.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn"))

total_depois = df_dedup.count()
duplicados_removidos = total_antes - total_depois
print(f"Registros depois da deduplicação: {total_depois}")
print(f"Duplicados removidos: {duplicados_removidos}")

# Sobrescrever a tabela Silver sem os duplicados
df_dedup.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("mvpDataEngineer.silver.medicao_laje")

print("Tabela mvpDataEngineer.silver.medicao_laje atualizada sem duplicatas.")

# Verificação: não deve haver mais duplicatas
w_check = Window.partitionBy("data_medicao", "cristalizador")
df_verificacao = (
    spark.table("mvpDataEngineer.silver.medicao_laje")
    .withColumn("qtd", count_fn(lit(1)).over(w_check))
    .filter(col("qtd") > 1)
)
restantes = df_verificacao.count()
print(f"Duplicatas restantes após deduplicação: {restantes}")

## 📝 Etapa 3: Documentação das Tabelas e Colunas

Comentários no Unity Catalog tornam os dados autodescritivos, visíveis no Catalog Explorer, autocomplete do SQL Editor e APIs.

In [0]:
%sql
-- =========================================
-- COMENTÁRIOS DA TABELA: leitura_pluviometros
-- =========================================

COMMENT ON TABLE mvpDataEngineer.silver.leitura_pluviometros IS 'Leituras pluviométricas diárias com tipos padronizados: datas como DATE, milímetros como DOUBLE, nomes de colunas em snake_case';

COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.unidade IS 'Unidade federativa de origem (ex: RN)';
COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.estacao IS 'Nome da estação pluviométrica';
COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.data_registro IS 'Data do registro da leitura (DATE)';
COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.milimetros IS 'Quantidade de chuva medida em milímetros (DOUBLE)';
COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.data_ingestao IS 'Timestamp da ingestão na camada Staging';
COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.origem IS 'Origem dos dados (google_drive)';
COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.arquivo_origem IS 'Nome do arquivo Excel de origem';
COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.data_carga_bronze IS 'Timestamp da carga na camada Bronze';
COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.origem_tabela IS 'Tabela de staging de origem';
COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.data_carga_silver IS 'Timestamp da carga na camada Silver';
COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.origem_tabela_bronze IS 'Tabela de bronze de origem';

In [0]:
%sql
-- =========================================
-- COMENTÁRIOS DA TABELA: medicao_bacias
-- =========================================

COMMENT ON TABLE mvpDataEngineer.silver.medicao_bacias IS 'Medições das bacias evaporadoras com tipos padronizados: volumes como INT, densidade como DOUBLE, valores ausentes como NULL';

COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.unidade IS 'Unidade federativa de origem (ex: RN)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.bacia IS 'Identificador da bacia evaporadora';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.data_registro IS 'Data do registro da medição (DATE)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.agua_doce_colocado IS 'Volume de água doce colocada na bacia (INT, NULL se ausente)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.agua_doce_retirado IS 'Volume de água doce retirada da bacia (INT, NULL se ausente)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.salmoura_colocado IS 'Volume de salmoura colocada na bacia (INT, NULL se ausente)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.salmoura_retirado IS 'Volume de salmoura retirada da bacia (INT, NULL se ausente)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.salmoura_densidade IS 'Densidade da salmoura em g/cm³ (DOUBLE)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.data_ingestao IS 'Timestamp da ingestão na camada Staging';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.origem IS 'Origem dos dados (google_drive)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.arquivo_origem IS 'Nome do arquivo Excel de origem';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.data_carga_bronze IS 'Timestamp da carga na camada Bronze';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.origem_tabela IS 'Tabela de staging de origem';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.data_carga_silver IS 'Timestamp da carga na camada Silver';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.origem_tabela_bronze IS 'Tabela de bronze de origem';

In [0]:
%sql
-- =========================================
-- COMENTÁRIOS DA TABELA: medicao_laje
-- =========================================

COMMENT ON TABLE mvpDataEngineer.silver.medicao_laje IS 'Medições da laje dos cristalizadores com tipos padronizados: data de medição como DATE, réguas como INT, nomes em snake_case';

COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.unidade IS 'Unidade federativa de origem (ex: RN)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.data_medicao IS 'Data da medição na laje (DATE)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.cristalizador IS 'Número de identificação do cristalizador';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.area_hectares IS 'Área do cristalizador em hectares (DOUBLE)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.regua_1 IS 'Leitura da régua 1 de medição de nível (INT)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.regua_2 IS 'Leitura da régua 2 de medição de nível (INT)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.regua_3 IS 'Leitura da régua 3 de medição de nível (INT)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.regua_4 IS 'Leitura da régua 4 de medição de nível (INT)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.regua_5 IS 'Leitura da régua 5 de medição de nível (INT)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.regua_6 IS 'Leitura da régua 6 de medição de nível (INT)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.regua_7 IS 'Leitura da régua 7 de medição de nível (INT)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.regua_8 IS 'Leitura da régua 8 de medição de nível (INT)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.data_ingestao IS 'Timestamp da ingestão na camada Staging';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.origem IS 'Origem dos dados (google_drive)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.arquivo_origem IS 'Nome do arquivo Excel de origem';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.data_carga_bronze IS 'Timestamp da carga na camada Bronze';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.origem_tabela IS 'Tabela de staging de origem';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.data_carga_silver IS 'Timestamp da carga na camada Silver';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.origem_tabela_bronze IS 'Tabela de bronze de origem';